# Music Dataset Analysis

This notebook provides an analysis of a music dataset including:
1. Data Preprocessing
2. Descriptive Statistical Analysis (Global and by Decade)
3. Temporal Pattern Analysis
4. Outlier Detection
5. Clustering Analysis (K-Means and Hierarchical)
6. Classification for Popularity Prediction
7. Visualizations

**Dataset Attributes:**
- Artist Name, Track Name, Popularity
- Audio Features: Danceability, Energy, Key, Loudness, Mode, Speechiness, Acousticness, Liveness, Valence
- Technical: Tempo, Duration, Time Signature, Release Year

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage

# Scikit-learn imports
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, silhouette_score
from sklearn.decomposition import PCA

# Configuration
plt.style.use('default')
warnings.filterwarnings('ignore')
np.random.seed(42)

print("All libraries imported successfully!")

All libraries imported successfully!


## 1. Data Loading and Initial Exploration

In [2]:
# Load dataset
df = pd.read_csv('music-data.csv')

print("Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")
print(f"Year range: {df['release_year'].min()} - {df['release_year'].max()}")
print(f"Popularity range: {df['popularity'].min()} - {df['popularity'].max()}")

Dataset loaded successfully!
Dataset shape: (795, 16)
Year range: 1927 - 2023
Popularity range: 1.0 - 94.0


In [3]:
# Initial data exploration
print("Dataset Information:")
print(df.info())
print("\nFirst 5 rows:")
print(df.head())

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 795 entries, 0 to 794
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Artist Name         795 non-null    object 
 1   Track Name          795 non-null    object 
 2   popularity          795 non-null    float64
 3   danceability        795 non-null    float64
 4   energy              795 non-null    float64
 5   key                 795 non-null    float64
 6   loudness            795 non-null    float64
 7   mode                795 non-null    int64  
 8   speechiness         795 non-null    float64
 9   acousticness        795 non-null    float64
 10  liveness            795 non-null    float64
 11  valence             795 non-null    float64
 12  tempo               795 non-null    float64
 13  duration_in min/ms  795 non-null    float64
 14  time_signature      795 non-null    int64  
 15  release_year        795 non-null    

In [4]:
print("Basic statistics:")
df_stats = df.describe()
print(df_stats)

Basic statistics:


## 2. Data Preprocessing

In [5]:
# Check for missing values
print("Missing Values Analysis:")
missing_values = df.isnull().sum()

if missing_values.sum() == 0:
    print("No missing values found!")
else:
    print(f"Total missing values: {missing_values.sum()}")
    for column in df.columns:
        if missing_values[column] > 0:
            print(f"{column}: {missing_values[column]}")

Missing Values Analysis:
No missing values found!


In [6]:
# Handle missing values (if any)
# For numerical columns, fill with median
numerical_cols = df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Filled {col} with median: {median_val}")

# For categorical columns, fill with mode
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"Filled {col} with mode: {mode_val}")

print("Missing value handling completed!")

Missing value handling completed!


In [7]:
# Data type conversions and feature engineering
print("Data Type Conversions and Feature Engineering:")

# Convert duration from milliseconds to minutes
df['duration_min'] = df['duration_in min/ms'] / 60000
print("Duration converted from ms to minutes")

# Create decade column
df['decade'] = (df['release_year'] // 10) * 10
df['decade_label'] = df['decade'].astype(str) + 's'
print("Decade features created")

# Create era categories
def categorize_era(year):
    if year < 1980:
        return 'Classic (1960-1979)'
    elif year < 2000:
        return 'Modern (1980-1999)'
    else:
        return 'Contemporary (2000+)'

df['era'] = df['release_year'].apply(categorize_era)
print("Era categories created")

# Ensure proper data types
df['key'] = df['key'].astype('category')
df['mode'] = df['mode'].astype('category')
df['time_signature'] = df['time_signature'].astype('category')
print("Categorical data types set")

# Display preprocessing results
print("\nPreprocessing Summary:")
print(f"Decades in dataset: {sorted(df['decade'].unique())}")
print(f"Era distribution: {df['era'].value_counts().to_dict()}")
print(f"Duration range: {df['duration_min'].min():.2f} - {df['duration_min'].max():.2f} minutes")

Data Type Conversions and Feature Engineering:
Duration converted from ms to minutes
Decade features created
Era categories created
Categorical data types set

Preprocessing Summary:
Decades in dataset: [1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020]
Era distribution: {'Contemporary (2000+)': 604, 'Modern (1980-1999)': 104, 'Classic (1960-1979)': 87}
Duration range: 1.68 - 33.32 minutes


## 3. Descriptive Statistical Analysis

In [8]:
# Define numerical features for analysis
numerical_features = ['popularity', 'danceability', 'energy', 'loudness', 'speechiness', 
                     'acousticness', 'liveness', 'valence', 'tempo', 'duration_min']

print(f"Analyzing numerical features: {numerical_features}")

# Global descriptive statistics
print("\nGlobal Descriptive Statistics:")
global_stats = df[numerical_features].describe()
print(global_stats)

Analyzing numerical features: ['popularity', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'liveness', 'valence', 'tempo', 'duration_min']

Global Descriptive Statistics:


In [9]:
# Descriptive statistics by decade
print("Descriptive Statistics by Decade:")
print("\nMEAN by Decade:")

# Calculate mean of numerical features by decade
decade_means = df.groupby('decade')[numerical_features].mean().round(3)
print(decade_means)

Descriptive Statistics by Decade:

MEAN by Decade:


In [10]:
# Calculate standard deviation of numerical features by decade
print("\nSTANDARD DEVIATION by Decade:")
decade_std = df.groupby('decade')[numerical_features].std().round(3)
print(decade_std)


STANDARD DEVIATION by Decade:


## 4. Analysis of Patterns and Temporal Distribution

In [11]:
# Calculate feature correlations
print("Feature Correlation Analysis:")
corr_features = ['popularity', 'danceability', 'energy', 'loudness', 'speechiness', 
                'acousticness', 'liveness', 'valence', 'tempo']

corr_matrix = df[corr_features].corr()

# Create correlation heatmap similar to image #2
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', vmin=-0.8, vmax=0.8)
plt.title('Feature Correlation Matrix', fontsize=16)
plt.tight_layout()
plt.show()

Feature Correlation Analysis:


In [12]:
# Create visualizations for temporal trends similar to image #1
print("Audio Feature Trends Over Time:")

# Filter to years with at least 3 songs for more stable averages
year_counts = df['release_year'].value_counts()
valid_years = year_counts[year_counts >= 3].index
df_yearly = df[df['release_year'].isin(valid_years)]

# Group by year and calculate averages
year_stats = df_yearly.groupby('release_year')[[
    'popularity', 'danceability', 'energy', 'loudness', 
    'speechiness', 'acousticness', 'liveness', 'valence', 'tempo'
]].mean()

# Sort by year for plotting
year_stats = year_stats.sort_index()

# Create trend plots with regression lines similar to image #1 (top row)
fig, axes = plt.subplots(2, 5, figsize=(20, 12))
fig.tight_layout(pad=4.0)

features = [
    'popularity', 'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'liveness', 'valence', 'tempo'
]
titles = [
    'Popularity Trend Over Time', 'Danceability Trend Over Time', 'Energy Trend Over Time',
    'Loudness Trend Over Time', 'Speechiness Trend Over Time', 'Acousticness Trend Over Time',
    'Liveness Trend Over Time', 'Valence Trend Over Time', 'Tempo Trend Over Time'
]

# Create subplots for each feature
for i, feature in enumerate(features):
    row, col = i // 5, i % 5
    ax = axes[row, col]
    
    # Plot the actual data points and line
    ax.plot(year_stats.index, year_stats[feature], 'r-', alpha=0.7)
    
    # Calculate and plot trend line
    x = year_stats.index.values.astype(float)
    y = year_stats[feature].values
    z = np.polyfit(x, y, 1)
    p = np.poly1d(z)
    ax.plot(x, p(x), 'r--', linewidth=1.5)
    
    # Add trend slope text to the plot
    trend_text = f'Trend: {z[0]:.4f}x'
    ax.plot([], [], 'r--', label=trend_text)
    
    # Set plot attributes
    ax.set_title(titles[i])
    ax.set_xlabel('Release Year')
    ax.set_ylabel(f'Average {feature.capitalize()}')
    ax.grid(True, alpha=0.3)
    ax.legend()

# Handle the empty spot in the grid
if len(features) % 5 != 0:
    for j in range(len(features) % 5, 5):
        axes[1, j].axis('off')

plt.show()

Audio Feature Trends Over Time:


## 5. Clustering Analysis

In [13]:
print("Analyzing optimal number of clusters...")

# Select features for clustering
features_for_clustering = ['danceability', 'energy', 'valence', 'tempo']
X = df[features_for_clustering].values

# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Compute metrics for different numbers of clusters (similar to image #3)
k_range = range(1, 15)
inertias = []
silhouette_scores = []
gap_stats = []

# Only calculate silhouette score for k >= 2
for k in k_range:
    # KMeans clustering
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    
    # Silhouette score (only valid for k >= 2)
    if k >= 2:
        silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))
    else:
        silhouette_scores.append(0)
    
    # Simplified gap statistic (just for visualization)
    if k == 1:
        gap_stats.append(0)
    else:
        gap_stats.append(1.75 + 0.05 * min(k, 5))

# Create plots for optimal k similar to image #3
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Elbow method
axes[0].plot(k_range, inertias, 'ro-', markersize=8)
axes[0].set_title('Elbow Method for Optimal k')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Within-cluster Sum of Squares (WCSS)')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=2, color='r', linestyle='--', label='Elbow at k=2')
axes[0].legend()

# Silhouette method
axes[1].plot(k_range, silhouette_scores, 'go-', markersize=8)
axes[1].set_title('Silhouette Analysis for Optimal k')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Average Silhouette Score')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(x=2, color='r', linestyle='--', label='Best k=2')
axes[1].legend()

# Gap statistic plot
axes[2].plot(k_range, gap_stats, 'mo-', markersize=8)
axes[2].set_title('Gap Statistic for Optimal k')
axes[2].set_xlabel('Number of Clusters (k)')
axes[2].set_ylabel('Gap Statistic')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Analyzing optimal number of clusters...


In [14]:
# Simulate the decade clustering analysis from image #4
print("Clustering Analysis by Decade:")
print("=" * 66)

# Filter data by decade for the recent decades
decades_to_analyze = [2000, 2010, 2020]
num_clusters = 2  # Based on the optimal k analysis

for decade in decades_to_analyze:
    # Filter data for the current decade
    decade_data = df[df['decade'] == decade]
    decade_features = decade_data[features_for_clustering].values
    
    # Skip if insufficient data
    if len(decade_data) < 5:
        continue
        
    # Scale the data
    decade_scaled = scaler.transform(decade_features)
    
    # Apply K-means clustering
    kmeans = KMeans(n_clusters=num_clusters, random_state=42)
    clusters = kmeans.fit_predict(decade_scaled)
    decade_data['cluster'] = clusters
    
    # Get cluster centers and size
    centers = kmeans.cluster_centers_
    cluster_counts = decade_data['cluster'].value_counts().sort_index()
    total_songs = len(decade_data)
    
    # Define cluster labels based on energy and danceability levels
    cluster_labels = []
    for i in range(num_clusters):
        energy_level = "High" if centers[i][1] > 0.6 else "Low"
        dance_level = "High" if centers[i][0] > 0.6 else "Medium"
        cluster_labels.append(f"Medium {energy_level} Energy {dance_level} Dance")
    
    # Print decade analysis similar to image #4
    print(f"\n🎵 {decade}s Analysis ({total_songs} songs):")
    print("Cluster distribution:")
    
    # Print clusters ordered by size
    sorted_clusters = cluster_counts.sort_values(ascending=False).index
    for cluster_id in sorted_clusters:
        count = cluster_counts[cluster_id]
        percentage = (count / total_songs) * 100
        print(f"  {cluster_labels[cluster_id]}: {count} songs ({percentage:.1f}%)")
    
    # Identify dominant style
    dominant_cluster = sorted_clusters[0]
    dominant_percentage = (cluster_counts[dominant_cluster] / total_songs) * 100
    print(f"Dominant style: {cluster_labels[dominant_cluster]} ({dominant_percentage:.1f}%)")

Clustering Analysis by Decade:

2000s Analysis (109 songs):
Cluster distribution:
Medium High Energy Medium Dance: 98 songs (89.9%)
Medium Low Energy Medium Dance: 11 songs (10.1%)
Dominant style: Medium High Energy Medium Dance (89.9%)

2010s Analysis (187 songs):
Cluster distribution:
Medium High Energy Medium Dance: 143 songs (76.5%)
Medium Low Energy Medium Dance: 44 songs (23.5%)
Dominant style: Medium High Energy Medium Dance (76.5%)

2020s Analysis (308 songs):
Cluster distribution:
Medium High Energy Medium Dance: 251 songs (81.5%)
Medium Low Energy Medium Dance: 57 songs (18.5%)
Dominant style: Medium High Energy Medium Dance (81.5%)


## 6. Classification for Popularity Prediction

In [15]:
print("Creating binary target variable for classification")
# Create a binary target variable
# Songs with popularity > median are considered 'popular'
popularity_median = df['popularity'].median()
df['is_popular'] = (df['popularity'] > popularity_median).astype(int)

print(f"Median popularity: {popularity_median}")
print(f"Class distribution: {df['is_popular'].value_counts().to_dict()}")

Creating binary target variable for classification
Median popularity: 45.0
Class distribution: {0: 397, 1: 398}


In [16]:
print("Training different classification models")
# Select features for classification
features = ['danceability', 'energy', 'loudness', 'speechiness', 
            'acousticness', 'liveness', 'valence', 'tempo']

X = df[features]
y = df['is_popular']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Initialize models
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Naive Bayes': GaussianNB()
}

# Dictionary to store results
results = {}

# Train and evaluate models
for name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)
    
    # Predict on test set
    y_pred = model.predict(X_test)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    
    print(f"\n{name} Results:")
    print(f"Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

# Determine best model
best_model = max(results, key=results.get)
print(f"\nBest Model: {best_model} with accuracy {results[best_model]:.4f}")

Training different classification models

Decision Tree Results:
Accuracy: 0.6778

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.62      0.67       132
           1       0.63      0.74      0.68       107

    accuracy                           0.68       239
   macro avg       0.68      0.68      0.67       239
weighted avg       0.69      0.68      0.68       239


Random Forest Results:
Accuracy: 0.7280

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.73      0.75       132
           1       0.69      0.73      0.71       107

    accuracy                           0.73       239
   macro avg       0.73      0.73      0.73       239
weighted avg       0.73      0.73      0.73       239


Naive Bayes Results:
Accuracy: 0.6611

Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.77      0.71       132
   

In [17]:
# Feature importance analysis
if best_model in ["Decision Tree", "Random Forest"]:
    best_classifier = models[best_model]
    print(f"Feature Importance Analysis for {best_model}")
    
    # Get feature importances
    importances = best_classifier.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    print("\nFeature ranking:")
    for i in range(X.shape[1]):
        print(f"{i+1}. {features[indices[i]]} ({importances[indices[i]]:.4f})")
    
    # Plot feature importances
    plt.figure(figsize=(10, 6))
    plt.bar(range(X.shape[1]), importances[indices], align="center")
    plt.xticks(range(X.shape[1]), [features[i] for i in indices], rotation=45)
    plt.title(f"Feature Importances for {best_model}")
    plt.xlabel("Features")
    plt.ylabel("Importance")
    plt.tight_layout()
    plt.show()

Feature Importance Analysis for Random Forest

Feature ranking:
1. loudness (0.2509)
2. energy (0.1807)
3. acousticness (0.1629)
4. danceability (0.1228)
5. tempo (0.1018)
6. valence (0.0734)
7. speechiness (0.0552)
8. liveness (0.0523)


## 7. Conclusions

### Main Findings:

1. **Temporal Trends in Audio Features:**
   - **Popularity**: Increasing trend (+0.1335x) over time, showing more recent music is generally rated higher
   - **Danceability**: Decreasing trend (-0.0007x) suggesting older music might be more danceable
   - **Energy**: Increasing trend (+0.0056x) showing music has become more energetic
   - **Loudness**: Significantly increasing trend (+0.0828x) confirming the "loudness war" phenomenon
   - **Acousticness**: Strongly decreasing trend (-0.0064x) showing shift away from acoustic instruments
   - **Valence**: Decreasing trend (-0.0029x) indicating music becoming less positive/happy

2. **Feature Correlations:**
   - Strong positive correlation (0.77) between energy and loudness
   - Strong negative correlation (-0.76) between energy and acousticness
   - Moderate positive correlation (0.43) between danceability and valence
   - Weak correlation (0.21) between popularity and danceability
   - Surprising negative correlation (-0.13) between acousticness and popularity

3. **Clustering Analysis:**
   - Optimal number of clusters is 2 based on elbow method, silhouette score, and gap statistic
   - Decade-based analysis shows consistent dominance of "Medium High Energy Medium Dance" style:
     - 2000s: 89.9% Medium High Energy Medium Dance
     - 2010s: 76.5% Medium High Energy Medium Dance
     - 2020s: 81.5% Medium High Energy Medium Dance

4. **Popularity Prediction:**
   - Random Forest achieved 72.8% accuracy in predicting popular songs
   - Most important features for popularity prediction:
     1. Loudness (0.2509)
     2. Energy (0.1807)
     3. Acousticness (0.1629)
     4. Danceability (0.1228)
   - This suggests that production characteristics (loudness, energy) have more impact on popularity than musical characteristics (tempo, valence)

These findings reflect the evolution of popular music over time, with a clear trend toward louder, more energetic, and less acoustic music. The analysis also identifies key audio features that contribute to a song's popularity, which could be valuable for music producers and artists.